# Kernel SVM

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
Some data cannot be separated by a straight line (say a circle inside another circle). The idea: lift the data to a **higher space** where it becomes separable. With a new feature $z=x_1^2+x_2^2$ (the distance from the center), the inner and outer circles are split by a simple plane $z=c$.

## The kernel trick
In its dual form, SVM only needs **inner products** of points: $\mathbf x_i^\top\mathbf x_j$. Replace them with a **kernel** $K(\mathbf x_i,\mathbf x_j)$ and it is as if we worked in a huge (even infinite-dimensional) space, **without ever building that space**.

$$f(\mathbf x)=\sum_{i\in SV}\alpha_iy_iK(\mathbf x_i,\mathbf x)+b$$

| Kernel | Formula | Parameters |
|---|---|---|
| Linear | $\mathbf x^\top\mathbf x'$ | — |
| Polynomial | $(\gamma\,\mathbf x^\top\mathbf x'+r)^d$ | `degree`, `coef0` |
| RBF (Gaussian) | $\exp(-\gamma\Vert\mathbf x-\mathbf x'\Vert^2)$ | `gamma` |
| Sigmoid | $\tanh(\gamma\,\mathbf x^\top\mathbf x'+r)$ | Rarely used |

> [Three kernels on circular data: linear fails; polynomial and RBF succeed.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/svm_kernels.png)

*Three kernels on circular data: linear fails; polynomial and RBF succeed.*



In [ ]:
import numpy as np
from sklearn.datasets import make_circles
from sklearn.svm import SVC

X, y = make_circles(300, noise=0.08, factor=0.5, random_state=0)
print("linear on x1, x2       :", round(SVC(kernel="linear").fit(X, y).score(X, y), 3))
# Add the feature z = x1^2 + x2^2 by hand: now a linear SVM works
Z = np.c_[X, (X ** 2).sum(1)]
print("linear on x1, x2, z    :", round(SVC(kernel="linear").fit(Z, y).score(Z, y), 3))
print("RBF kernel on x1, x2   :", round(SVC(kernel="rbf").fit(X, y).score(X, y), 3))

# The RBF kernel value between two points
a, b = np.array([0, 0]), np.array([1, 1])
for g in [0.1, 1, 10]:
    print(f"K(a, b) with gamma={g}: {np.exp(-g * np.sum((a - b) ** 2)):.4f}")



## gamma and C together
- **gamma:** the reach of each support vector. Small = wide reach and a smooth boundary; large = local reach and a wiggly boundary.
- **C:** how much training errors cost.

[[lab:cls_ksvm]]

In the lab, make gamma very large: an "island" forms around each point (overfitting).

| | Small gamma | Large gamma |
|---|---|---|
| Small C | Very simple | Local but soft |
| Large C | Smooth and accurate | Severe overfitting |

> Always search gamma and C **together** on a log grid, e.g. $C\in\{0.1,1,10,100\}$ and $\gamma\in\{0.001,0.01,0.1,1\}$.



In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_breast_cancer(return_X_y=True)
grid = GridSearchCV(make_pipeline(StandardScaler(), SVC()), {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.001, 0.01, 0.1, 1]}, cv=5)
grid.fit(X, y)
print("Best:", grid.best_params_, round(grid.best_score_, 3))
table = pd.DataFrame(grid.cv_results_)[["param_svc__C", "param_svc__gamma", "mean_test_score"]]
print(table.pivot(index="param_svc__C", columns="param_svc__gamma", values="mean_test_score").round(3))



| Pros | Cons |
|---|---|
| Accurate non-linear boundaries | Slow beyond tens of thousands of samples |
| Works well in high dimensions (text) | Choosing the kernel and parameters |
| Small memory: only support vectors | Hard to interpret |

### Exercise

Compare the linear, poly and rbf kernels on make_moons.

In [ ]:
# Your answer here